初始化

In [ ]:
import re
import os
import joblib
from joblib import Parallel, delayed
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import linregress
import warnings
import numpy as np 
import pandas as pd
import polars as pl
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
import kaggle_evaluation.default_inference_server



import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


数据导入

In [ ]:
INPUT_PATH = '/kaggle/input/hull-tactical-market-prediction/'

# 这里后面修改 MODEL_PATH = '/tmp/lgbm_model.pkl'
TARGET_COL = 'market_forward_excess_returns'
EXCLUDE_COLS = ['date_id', 'forward_returns', 'risk_free_rate', TARGET_COL]
print("Loading data...")
train = pd.read_csv(os.path.join(INPUT_PATH, 'train.csv'))
print(f"Train shape: {train.shape}")
##warnings.filterwarning("ignore")


数据预处理：这里对波动率指标用了较大的篇幅进行处理，因为默认比较重要且难以简单处理

In [ ]:
def get_feature_group(prefix):
        return [c for c in train.columns if re.match(f'^{prefix}[0-9]+$', c)] ##正则筛选特定前缀的列名，“[0-9]+”匹配一或多个数字
groups = {
    'D': get_feature_group('D'),
    'E': get_feature_group('E'),
    'I': get_feature_group('I'),
    'M': get_feature_group('M'),
    'P': get_feature_group('P'),
    'S': get_feature_group('S'),
    'V': get_feature_group('V'),
    }



def _calculate_single_column_stress(
    vol_series: pd.Series,
    window: int
) -> pd.Series:
    """计算单列市场压力指数"""
    if vol_series.empty or vol_series.isna().all():
        return pd.Series(0.5, index=vol_series.index)
    
    # 1. 波动率历史分位数（核心）
    rolling_quantile = (
        vol_series
        .expanding(min_periods=window)
        .apply(lambda x: (x[-1] > x[:-1]).mean() if len(x) > 1 else 0.5, raw=False)
        .fillna(0.5)  # 初始值
    )
    
    # 2. 波动率斜率（加速上升）
    vol_slope = (
        vol_series
        .rolling(window=5, min_periods=3)
        .apply(lambda x: linregress(range(len(x)), x)[0] if len(x) > 3 else 0, raw=False)
    )
    vol_std = vol_series.rolling(window).std().fillna(1)
    vol_slope_norm = vol_slope / vol_std
    
    # 3. 融合压力指数
    stress_index = (
        rolling_quantile * 0.7 + 
        ((vol_slope_norm > 0) & (vol_slope_norm > vol_slope_norm.rolling(10).quantile(0.7))).astype(float) * 0.3
    )
    
    return stress_index.clip(0, 1).fillna(0.5)

def vol_midium_missing(
    vol_df: pd.DataFrame,
    pressure_window: int = 63,
    lookback_window: int = 21,
    parallel: bool = False,  # 默认关闭并行
    n_jobs: int = -1
) -> pd.DataFrame:
    """
    处理波动率特征的中等缺失值
    严格保持原始索引和列名
    """
    # 验证输入
    if not isinstance(vol_df, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame")
    if vol_df.empty:
        return vol_df.copy()
    
    original_index = vol_df.index.copy()
    original_columns = vol_df.columns.tolist()
    
    # 安全处理单列情况
    if len(original_columns) == 1:
        parallel = False
    
    def process_single_column(col_name: str) -> pd.Series:
        """处理单个波动率列，返回带原始索引的Series"""
        if col_name not in vol_df.columns:
            raise KeyError(f"Column '{col_name}' not found in input DataFrame")
        
        col_series = vol_df[col_name].copy()
        original_name = col_series.name

        col_series = col_series.replace([np.inf, -np.inf], np.nan) #消除除零错误等无穷值
        
        # 阶段1：时序局部填充
        if col_series.isna().all():
            print(f"⚠️ Column '{col_name}' is entirely NaN. Using zeros.")
            return pd.Series(0.0, index=col_series.index, name=original_name)
        safe_fill_value = col_series.median() if not col_series.isna().all() else 0.0
        safe_fill_value = max(0.0, safe_fill_value)  # 波动率不能为负
    
    # 创建临时序列：仅用于滚动计算
        temp_series = col_series.fillna(safe_fill_value)
        try:
            window_size = min(lookback_window, len(col_series) // 2 + 1)
            rolling_median = np.array([
            np.median(temp_series.iloc[max(0, i-window_size+1):i+1].values)
            for i in range(len(temp_series))
        ])
            rolling_median = pd.Series(rolling_median, index=col_series.index)
            # 仅用滚动中位数填充原始序列的NaN
            ewm_filled = col_series.copy()
            ewm_filled.loc[col_series.isna()] = rolling_median.loc[col_series.isna()]
        
            #   验证
            if ewm_filled.isna().all() or np.isinf(ewm_filled).any():
                raise ValueError("Rolling median produced invalid values")
            
            return ewm_filled
        except Exception as e:
        # 备选：指数加权平均
            try:
                ewm_filled = col_series.ewm(span=10).mean()
                return ewm_filled.ffill().bfill()  # 确保无 NaN
            except:
                # 最终回退
                print(f"Fallback for {col_name}: {str(e)}")
                ewm_filled=col_series.ffill().bfill()
                return ewm_filled   
        
        # 阶段2：计算市场压力
        market_stress = _calculate_single_column_stress(col_series, pressure_window)
        
        # 阶段3：全局统计填充（按压力分组）
        if ewm_filled.isna().any():
            stress_threshold = market_stress.quantile(0.7)
            high_stress_mask = market_stress > stress_threshold
            
            # 高压力区域：用90分位数
            high_stress_vol = col_series[high_stress_mask & ~col_series.isna()]
            high_fill_val = (
                high_stress_vol.quantile(0.9) if len(high_stress_vol) > 5 
                else col_series.quantile(0.75)
            )
            
            # 低压力区域：用50分位数
            low_stress_vol = col_series[~high_stress_mask & ~col_series.isna()]
            low_fill_val = (
                low_stress_vol.quantile(0.5) if len(low_stress_vol) > 5 
                else col_series.quantile(0.5)
            )
            
            # 应用填充
            ewm_filled.loc[high_stress_mask & ewm_filled.isna()] = high_fill_val
            ewm_filled.loc[~high_stress_mask & ewm_filled.isna()] = low_fill_val
        
        # 最终填充
        final_filled = ewm_filled.ffill().bfill()
        
        return pd.Series(
            data=final_filled.values,
            index=original_index,
            name=original_name,
            dtype=final_filled.dtype
        )
    
    # 处理所有列
    if parallel and len(original_columns) > 1:
        # 并行处理
        try:
            results = Parallel(n_jobs=n_jobs)(
                delayed(process_single_column)(col) for col in original_columns
            )
            # 重建DataFrame
            filled_data = {col: res for col, res in zip(original_columns, results)}
            result = pd.DataFrame(filled_data, index=original_index)
        except Exception as e:
            print(f"ParallelGroup processing failed: {str(e)}. Falling back to sequential.")
            parallel = False
    
    if not parallel or 'result' not in locals():
        # 顺序处理（更安全）
        filled_data = {}
        for col in original_columns:
            try:
                filled_data[col] = process_single_column(col)
            except Exception as e:
                print(f"Error processing column '{col}': {str(e)}")
                # 回退方案
                col_series = vol_df[col].copy()
                filled_data[col] = col_series.ffill().bfill()
        
        # 重建DataFrame
        result = pd.DataFrame(filled_data, index=original_index)
    
   
    result = result[original_columns]  # 保持原始列顺序
    result.index = original_index      # 确保索引正确
    
    return result

def handle_missing(train):
    """分层次与类别处理缺失值，全部的特征包括[D1,D2,D3,...D9   二进制特征
                  E1,E2,E3...E20  宏观经济特征
                  I1,I2,I3...I9    利率特征，滞后
                  M1,M2,M3...M18  市场动态技术特征，滞后
                  P1,P2,P3...P13  价格，估值特征
                  S1,S2,S3...S12   情感特写
                  V1,V2,V3...V13   波动率，滞后 
                  MOM,     动量特征 实际没有自己创建         ]
        高缺失率（>50%):直接删除，中等（5-50）使用时间序列特性填充，低（<5%)前向或者迭代填充
    """
    
    missing_ratio = train.isna().mean()
    too_missing = missing_ratio[missing_ratio > 0.5].index.tolist()
    train.drop(columns=too_missing, inplace=True, errors='ignore')

    low_missing_cols = set(missing_ratio[(missing_ratio > 0) & (missing_ratio < 0.05)].index.tolist())
    medium_missing_cols = set(missing_ratio[(missing_ratio >= 0.05) & (missing_ratio <= 0.5)].index.tolist())

    
    # 对低缺失率以及中缺失率数据按组进行缺失处理
    for gname, cols in groups.items():
        valid_cols1=set(cols) & low_missing_cols
        valid_cols2=set(cols) & medium_missing_cols

        for col in valid_cols1:
            
            if gname == 'D':
                train[col] = train[col].fillna(0)
            elif gname in ['E', 'I','P','V']:
                train[col] = train[col].ffill().fillna(train[col].median())
            elif gname == 'V':
                
                train[col] = train[col].fillna(train[col].rolling(window=21,min_periods=1).median()).ffill()
            elif gname == 'M':
                train[col] = train[col].ffill().fillna(train[col].mean())
            elif gname == 'S':
                col_series = train[col].copy()
                col_series = col_series.fillna(col_series.rolling(5, min_periods=1).mean())
                col_series = col_series.ffill(limit=5)
                col_series = col_series.ffill().fillna(col_series.mean())
                train[col] = col_series
        for col in valid_cols2:
            
            if gname == 'D':
                train[col] = train[col].fillna(0)
            elif gname in ['E', 'I','P']:
                train[col] = train[col].ffill().bfill()

            elif gname == 'V':
                col_df = train[[col]].ffill()
                filled_col = vol_midium_missing(
                    col_df, 
                    pressure_window=63,
                    lookback_window=21,
                    parallel=False

                )
                train[col] = filled_col[col]
                # train[cols] = vol_midium_missing(train[cols].ffill())
            elif gname == 'M':
                train[col] = train[col].ffill().fillna(train[col].mean())
            elif gname == 'S':
                col_series = train[col].copy()
                col_series = col_series.fillna(col_series.rolling(5, min_periods=1).mean())
                col_series = col_series.ffill(limit=5)
                col_series = col_series.ffill().fillna(col_series.mean())
                train[col] = col_series
             
    return train


def handle_outliers(df, clip=1):
    """处理异常值"""
    numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns 
                    if c not in EXCLUDE_COLS]
    for col in numeric_cols:
        low = df[col].quantile(clip/100)
        high = df[col].quantile(1-clip/100)
        df[col] = df[col].clip(low, high)
    return df

print("Preprocessing training data...")
# train = train.iloc[1005:].reset_index(drop=True)

train = handle_missing(train)
train = handle_outliers(train)

print(f"Train shape: {train.shape}")

In [ ]:
print(f"nonsum:{train.notnull().sum()}")

特征工程，这里的动量特征做出来以后发现有过拟合的风险，本来利用forward_returns滞后计算后进行创建，后删除，对情绪指标进行滞后创建以及后续再考虑特征工程进行分组创建新特征

In [ ]:
###初步方案，只涉及部分
def create_finance_features(df):
    """创建特征，其中根据EDA的数据集，特征与目标斯皮尔曼相关系数以及共线性分析
    全部的特征包括[D1,D2,D3,...D9   二进制特征
                  E1,E2,E3...E20  宏观经济特征
                  I1,I2,I3...I9    利率特征
                  M1,M2,M3...M18  市场动态技术特征，滞后
                  P1,P2,P3...P13  价格，估值特征
                  S1,S2,S3...S12   情感特写,短期滞后，123加权
                  V1,V2,V3...V13   波动率，滞后
                  创建mom动量特征，mom5，mom21           ]
    其中，前十和最后十名的相关特征分别为[M1,V13,V10,S5,V7,E19,V9,S12,D2,D1],[E12,S8,P8,M12,I2,S3,E11,P5,S2,M4]
    共线性最高的十位特征为[I5,I9,M14,E2,P10,E3,I7,P11,P8,I8]
    同时我们通过检验证明了该时间序列基本满足平稳性(stationary)
    """
    
    #关于滞后特征
    '''
    # 动量特征
    df['mom_5d'] = df[TARGET_COL].rolling(window=5, min_periods=1).sum()
    df['mom_21d'] = df[TARGET_COL].rolling(window=21, min_periods=1).sum()
    
    for gname , cols in groups.items():
        if gname =='S':
            for col in cols:
                shifted_col = df[col].shift(1)
            
            # 计算加权滞后特征（只使用历史数据）
                df[f"lag_{col}"] = (
                    0.5 * shifted_col +  
                    0.3 * shifted_col.rolling(window=3, min_periods=1).mean() +  
                    0.2 * shifted_col.rolling(window=5, min_periods=1).mean()    
                )
                
                del shifted_col

            
        elif gname == 'M':

        elif gname == 'V':
        '''
        #这里被提示了性能问题，内存碎片化严重，因为循环中多次插入新列df[f"lag_{col}]=...，因此采取优化
        #方案如下
      # === 1. 定义所有需要管理的特征列名 ===
    # 情绪特征 (lag_)
    lag_features = [
        f"lag_{col}" 
        for col in groups.get('S', []) 
        if col in df.columns
    ]
    
    # 动量特征 (固定名称)
    df['hist_returns'] = df['forward_returns'].shift(1)
    mom_features = ['mom_21d', 'mom_42d','mom_63d','mom_126']
    
    # 合并所有需要管理的特征
    all_managed_features = lag_features + mom_features
    
    # === 2. 删除所有旧特征（幂等性核心） ===
    existing_features = [f for f in all_managed_features if f in df.columns]
    if existing_features:
        df = df.drop(columns=existing_features)
        print(f"Removed {len(existing_features)} outdated features: {existing_features}")
    
    # === 3. 批量计算新特征 ===
    new_features = {}
    
    # 动量特征（使用清理后的数据计算）
   
    #new_features['mom_21d'] = df['hist_returns'] .rolling(window=21, min_periods=1).sum()
    #new_features['mom_42d'] = df['hist_returns'] .rolling(window=42, min_periods=1).sum()
    #new_features['mom_63d'] = df['hist_returns'] .rolling(window=63, min_periods=1).sum()
    #new_features['mom_126'] = df['hist_returns'] .rolling(window=126, min_periods=1).sum()
    # 情绪特征
    for col in groups.get('S', []):
        if col not in df.columns:
            continue
            
        shifted = df[col].shift(1)
        new_features[f"lag_{col}"] = (
            0.5 * shifted +
            0.3 * shifted.rolling(3, min_periods=1).mean() +
            0.2 * shifted.rolling(5, min_periods=1).mean()
        )
    
    # === 4. 一次性合并所有新特征 ===
    if new_features:
        features_df = pd.DataFrame(new_features, index=df.index)
        df = pd.concat([df, features_df], axis=1).ffill().bfill()
    
    return df

train=create_finance_features(train)

print(train.shape)
print(train.head)
        

In [ ]:
new_groups = {
    'D': get_feature_group('D'),
    'E': get_feature_group('E'),
    'I': get_feature_group('I'),
    'M': get_feature_group('M'),
    'P': get_feature_group('P'),
    'S': get_feature_group('S'),
    'V': get_feature_group('V'),
    }
def create_lag_features_efficient(df, groups, lags=[1, 3]):
    
    # 1. 确定需要创建滞后的组
    lag_groups = ['I', 'M', 'V','S']  # 仅对这四组创建滞后特征
    
    # 2. 为所有新特征创建字典
    lag_features = {}
    
    print("\n=== 高效创建滞后特征 ===")
    for group_name in lag_groups:
        if group_name not in groups:
            continue
            
        cols = groups[group_name]
        print(f"处理 {group_name} 组: {len(cols)} 列")
        

        if group_name == 'S':    
            shifted = df[col].shift(1)
            lag_features[f"lag_{col}"] = (
                0.5 * shifted +
                0.3 * shifted.rolling(3, min_periods=1).mean() +
                0.2 * shifted.rolling(5, min_periods=1).mean()
            )
        else:
            for col in tqdm(cols, desc=f"{group_name}组滞后"):
                for lag in lags:
                    # 计算滞后特征并存储到字典
                    lag_features[f'{col}_lag{lag}'] = df[col].shift(lag)
    
    # 3. 将字典转换为DataFrame
    lag_df = pd.DataFrame(lag_features, index=df.index)
    
    # 4. 一次性合并到原DataFrame
    result_df = pd.concat([df, lag_df], axis=1).ffill().bfill()
    
    print(f"成功添加 {len(lag_df.columns)} 个滞后特征")
    return result_df
    
def create_finance_features(df, create_lags=True, max_lag=5):
    print("\n=== 特征分组验证 ===")
    for gname, cols in groups.items():
        print(f"{gname}组特征数: {len(cols)}")
    
    # 4. 创建滞后特征 (仅针对I, M, V, S组)
    df = create_lag_features_efficient(df.copy(), new_groups, lags=[1, 3])
    # 5. 组内统计特征 (安全，无假设)
    print("\n=== 创建组内统计特征 ===")
    
    for group_name, cols in new_groups.items():
        if not cols:
            continue
        
        # 5.1 组均值/标准差
        df[f'{group_name}_mean'] = df[cols].mean(axis=1)
        df[f'{group_name}_std'] = df[cols].std(axis=1)
        
        # 5.2 组内极差
        df[f'{group_name}_range'] = df[cols].max(axis=1) - df[cols].min(axis=1)
        
        # 5.3 非二进制特征添加变异系数
        if group_name != 'D':
            df[f'{group_name}_cv'] = df[f'{group_name}_std'] / (df[f'{group_name}_mean'].abs() + 1e-8)
    
    # 6. 组间交互特征 (基于统计相关性)
    print("\n=== 创建组间交互特征 ===")
    
    # 6.1 技术指标与波动率交互 (市场信号强度 = 技术指标/波动率)
    if 'M_mean' in df.columns and 'V_mean' in df.columns:
        df['tech_vol_strength'] = df['M_mean'] / (df['V_mean'] + 1e-8)
    
    # 6.2 利率与市场技术交互 (利率环境对技术信号的影响)
    if 'I_mean' in df.columns and 'M_mean' in df.columns:
        df['rate_tech_interaction'] = df['I_mean'] * df['M_mean']
    
    # 6.3 价格与波动率关系 (风险调整价格信号)
    if 'P_mean' in df.columns and 'V_mean' in df.columns:
        df['price_vol_adjusted'] = df['P_mean'] / (df['V_mean'] + 1e-8)
    
    # 6.4 宏观经济与利率交互 (货币环境信号)
    if 'E_mean' in df.columns and 'I_mean' in df.columns:
        df['macro_rate_interaction'] = df['E_mean'] * (1 / (df['I_mean'] + 1e-8))
    
    # 7. 滚动统计特征 (严格时间序列安全)
    print("\n=== 创建滚动统计特征 ===")
    
    # 选择高价值核心特征
    core_features = []
    for group in ['M', 'V', 'I', 'E', 'P']:
        if f'{group}_mean' in df.columns:
            core_features.append(f'{group}_mean')
    
    # 添加已验证的交互特征
    interaction_features = ['tech_vol_strength', 'price_vol_adjusted']
    core_features.extend([f for f in interaction_features if f in df.columns])
    core_features = list(set(core_features))
    
    windows = [5, 10]  # 适应小数据集的窗口
    for window in windows:
        for feature in tqdm(core_features, desc=f"滚动窗口{window}"):
            # 滚动均值 (shift(1)确保安全)
            df[f'{feature}_roll{window}_mean'] = (
                df[feature].rolling(window, min_periods=1).mean().shift(1)
            )
            
            # 滚动标准差 (仅对非波动特征)
            if 'vol' not in feature.lower() and 'std' not in feature.lower():
                df[f'{feature}_roll{window}_std'] = (
                    df[feature].rolling(window, min_periods=1).std().shift(1)
                )
    
    # 8. 内存优化
    print("\n=== 优化内存使用 ===")
    # 二进制特征压缩
    for col in new_groups['D']:
        if col in df.columns:
            df[col] = df[col].astype(np.int8)
    
    # 浮点列压缩
    float_cols = df.select_dtypes(include=['float64']).columns
    df[float_cols] = df[float_cols].astype(np.float32)
    
    # 整数列压缩
    int_cols = df.select_dtypes(include=['int64']).columns
    df[int_cols] = df[int_cols].astype(np.int16)
    
    # 9. 特征验证
    print("\n=== 特征工程结果 ===")
    total_features = len(df.columns)
    original_features = sum(len(cols) for cols in new_groups.values()) + (1 if 'data_ID' in df.columns else 0)
    new_features = total_features - original_features
    
    print(f"原始特征数: {original_features}")
    print(f"新增特征数: {new_features}")
    print(f"总特征数: {total_features}")
    
    return df

train=create_finance_features(train).ffill().bfill()
print(train.shape)
print(train.head)


In [ ]:
def create_finance_features(df, create_lags=True, max_lag=5):
    print("\n=== 特征分组验证 ===")
    original_columns = set(df.columns)  # 保存原始列名
    
    # 1. 重新定义特征分组 (仅使用原始特征)
    def get_base_feature_group(prefix):
        """仅匹配原始特征 (不含衍生特征)"""
        return [c for c in original_columns if re.fullmatch(f'^{prefix}[0-9]+$', c)]
    
    base_groups = {
        'D': get_base_feature_group('D'),
        'E': get_base_feature_group('E'),
        'I': get_base_feature_group('I'),
        'M': get_base_feature_group('M'),
        'P': get_base_feature_group('P'),
        'S': get_base_feature_group('S'),
        'V': get_base_feature_group('V'),
    }
    
    for gname, cols in base_groups.items():
        print(f"{gname}组特征数: {len(cols)} (示例: {cols[:3] if cols else '无'})")
    
    # 2. 创建滞后特征 (仅针对指定组)
    if create_lags:
        # 2.1 仅对原始特征创建滞后
        existing_lag_cols = [c for c in df.columns if '_lag' in c or 'lag_' in c]
        print(f"\n检测到 {len(existing_lag_cols)} 个已存在的滞后特征，跳过重复创建")
        
        # 2.2 只创建不存在的滞后特征
        lag_groups = ['I', 'M', 'V', 'S']
        lag_features = {}
        new_lag_count = 0
        
        print("\n=== 创建滞后特征 ===")
        for group_name in lag_groups:
            cols = base_groups.get(group_name, [])
            if not cols:
                continue
            
            print(f"处理 {group_name} 组原始特征: {len(cols)} 列")
            for col in tqdm(cols, desc=f"{group_name}组滞后"):
                if group_name == 'S':
                    # 情绪特征特殊处理
                    if f'lag_{col}' not in df.columns:
                        shifted = df[col].shift(1)
                        lag_features[f"lag_{col}"] = (
                            0.5 * shifted +
                            0.3 * shifted.rolling(3, min_periods=1).mean() +
                            0.2 * shifted.rolling(5, min_periods=1).mean()
                        )
                        new_lag_count += 1
                else:
                    for lag in [1, 3]:  # 固定滞后阶数
                        new_col = f'{col}_lag{lag}'
                        if new_col not in df.columns:
                            lag_features[new_col] = df[col].shift(lag)
                            new_lag_count += 1
        
        # 2.3 合并新滞后特征
        if lag_features:
            lag_df = pd.DataFrame(lag_features, index=df.index)
            df = pd.concat([df, lag_df], axis=1).ffill().bfill()
            print(f"成功添加 {len(lag_df.columns)} 个新滞后特征 (跳过 {new_lag_count - len(lag_df.columns)} 个已存在特征)")
        else:
            print("未添加新滞后特征 (所有特征已存在)")
    
    # 3. 组内统计特征 (防止重复)
    print("\n=== 创建组内统计特征 ===")
    for group_name, cols in base_groups.items():
        if not cols:
            continue
        
        # 3.1 检查特征是否已存在
        mean_col = f'{group_name}_mean'
        std_col = f'{group_name}_std'
        range_col = f'{group_name}_range'
        cv_col = f'{group_name}_cv'
        
        # 3.2 仅创建不存在的特征
        if mean_col not in df.columns:
            df[mean_col] = df[cols].mean(axis=1)
        
        if std_col not in df.columns:
            df[std_col] = df[cols].std(axis=1)
        
        if range_col not in df.columns:
            df[range_col] = df[cols].max(axis=1) - df[cols].min(axis=1)
        
        if group_name != 'D' and cv_col not in df.columns:
            df[cv_col] = df[std_col] / (df[mean_col].abs() + 1e-8)
        
        print(f"处理 {group_name} 组: 已添加缺失的统计特征")
    
    # 4. 组间交互特征 (防止重复)
    print("\n=== 创建组间交互特征 ===")
    interactions = [
        ('tech_vol_strength', ['M_mean', 'V_mean'], lambda x, y: x / (y + 1e-8)),
        ('rate_tech_interaction', ['I_mean', 'M_mean'], lambda x, y: x * y),
        ('price_vol_adjusted', ['P_mean', 'V_mean'], lambda x, y: x / (y + 1e-8)),
        ('macro_rate_interaction', ['E_mean', 'I_mean'], lambda x, y: x * (1 / (y + 1e-8)))
    ]
    
    for feat_name, req_cols, func in interactions:
        if feat_name not in df.columns:
            if all(col in df.columns for col in req_cols):
                df[feat_name] = func(*[df[col] for col in req_cols])
                print(f"创建交互特征: {feat_name}")
            else:
                print(f"跳过 {feat_name}: 缺少必要列 {req_cols}")
        else:
            print(f"跳过已存在的交互特征: {feat_name}")
    
    # 5. 滚动统计特征 (防止重复)
    print("\n=== 创建滚动统计特征 ===")
    core_features = []
    for group in ['M', 'V', 'I', 'E', 'P']:
        mean_col = f'{group}_mean'
        if mean_col in df.columns:
            core_features.append(mean_col)
    
    interaction_features = ['tech_vol_strength', 'price_vol_adjusted']
    core_features.extend([f for f in interaction_features if f in df.columns])
    core_features = list(set(core_features))
    
    windows = [5, 10]
    new_roll_count = 0
    
    for window in windows:
        for feature in tqdm(core_features, desc=f"滚动窗口{window}"):
            roll_mean_col = f'{feature}_roll{window}_mean'
            roll_std_col = f'{feature}_roll{window}_std'
            
            # 仅创建不存在的滚动特征
            if roll_mean_col not in df.columns:
                df[roll_mean_col] = df[feature].rolling(window, min_periods=1).mean().shift(1)
                new_roll_count += 1
            
            # 仅对非波动特征创建标准差
            if 'vol' not in feature.lower() and 'std' not in feature.lower():
                if roll_std_col not in df.columns:
                    df[roll_std_col] = df[feature].rolling(window, min_periods=1).std().shift(1)
                    new_roll_count += 1
    
    print(f"添加 {new_roll_count} 个新滚动统计特征")
    
    # 6. 内存优化
    print("\n=== 优化内存使用 ===")
    # 仅对原始二进制特征进行压缩
    for col in base_groups['D']:
        if col in df.columns and df[col].dtype in ['int64', 'int32']:
            df[col] = df[col].astype(np.int8)
    
    # 浮点列压缩
    float_cols = df.select_dtypes(include=['float64']).columns
    for col in float_cols:
        if df[col].dtype == 'float64':
            df[col] = df[col].astype(np.float32)
    
    # 整数列压缩
    int_cols = df.select_dtypes(include=['int64']).columns
    for col in int_cols:
        max_val = df[col].max()
        if max_val < 32767:
            df[col] = df[col].astype('int16')
    
    # 7. 特征验证
    print("\n=== 特征工程结果 ===")
    # 仅计算新添加的特征
    current_cols = set(df.columns)
    new_cols = current_cols - original_columns
    print(f"原始特征数: {len(original_columns)}")
    print(f"新增特征数: {len(new_cols)}")
    print(f"总特征数: {len(df.columns)}")
    
    # 8. 处理缺失值 (仅在新特征上)
    print("\n=== 处理缺失值 ===")
    for col in new_cols:
        if df[col].isna().any():
            if df[col].dtype in ['float32', 'float64']:
                fill_val = df[col].median()
            else:
                fill_val = df[col].mode()[0] if not df[col].mode().empty else 0
            df[col].fillna(fill_val, inplace=True)
            print(f"填充 {col} 的缺失值: {fill_val}")
    
    return df

# 使用示例
train_processed = create_finance_features(train.copy())
print(f"\n处理后的数据形状: {train_processed.shape}")

# 验证幂等性
train_processed_again = create_finance_features(train_processed.copy())
print(f"再次处理后的数据形状: {train_processed_again.shape} (应与上次相同)")

这里把数据统一分组整理一下，后续进行特征选择以及模型和交叉验证

In [ ]:
drop_cols = ['hist_returns','date_id', 'forward_returns', 'risk_free_rate', TARGET_COL]
drop_cols = [c for c in drop_cols if c in train.columns]

n_test = 300
n_train = len(train) - n_test

score_forward_returns=pd.Series(train.iloc[n_train:]['forward_returns'])
score_risk_free_rate=pd.Series(train.iloc[n_train:]['risk_free_rate'])

#train.drop(columns=drop_cols, inplace=True, errors='ignore')

feature_cols = [col for col in train.columns if col not in drop_cols ]

x_train = train.iloc[:n_train, train.columns.isin(feature_cols)]
y_train = train.iloc[:n_train][TARGET_COL]
x_test = train.iloc[n_train:, train.columns.isin(feature_cols)]
y_test = train.iloc[n_train:][TARGET_COL]
train_score_forward_returns=pd.Series(train.iloc[:n_train]['forward_returns'])
train_score_risk_free_rate=pd.Series(train.iloc[:n_train]['risk_free_rate'])

print(f"nonsum:{x_test.notnull().sum()}")
print(f"nonsum:{x_train.notnull().sum()}")
print(y_train.head)

特征选择

In [ ]:
#最新的详细特征工程选择代码
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

import seaborn as sns
from sklearn.feature_selection import RFECV, SelectFromModel
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

class FinancialFeatureSelector:
    """
    金融时间序列安全的特征选择框架
    特点:
    1. 严格时间序列安全 (在TimeSeriesSplit内部执行所有特征选择)
    2. 分阶段筛选 (粗过滤->重要性排序->RFE->业务验证)
    3. 业务逻辑验证 (保留金融意义特征)
    4. 过拟合防护 (限制特征/样本比例)
    """
    
    def __init__(self, target_col='target', max_features=50, min_features=30):
        self.target_col = target_col
        self.max_features = max_features
        self.min_features = min_features
        self.selected_features = None
        self.feature_metadata = {}
    
    def preprocess_features(self, X):
        """处理特征，替换无穷大和填充缺失值"""
        X = X.copy()
        
        # 替换无穷大
        for col in X.select_dtypes(include=[np.number]).columns:
            X[col] = X[col].replace([np.inf, -np.inf], np.nan)
        
        # 用训练集统计数据填充缺失值
        for col in X.columns:
            if X[col].isna().any():
                if X[col].dtype in ['float32', 'float64']:
                    fill_value = X[col].median()
                else:
                    fill_value = X[col].mode()[0] if len(X[col].mode()) > 0 else 0
                X[col].fillna(fill_value, inplace=True)
        
        return X
    
    def pre_filter_features(self, X, y, corr_threshold=0.9):
        """
        预过滤: 移除明显无效特征
        1. 常量特征
        2. 高缺失率特征
        3. 高度相关特征
        """
        X = X.copy()
        
        # 1. 移除常量特征
        constant_cols = [col for col in X.columns if X[col].nunique() <= 1]
        print(f"移除 {len(constant_cols)} 个常量特征: {constant_cols[:5]}...")
        
        # 2. 移除高缺失率特征 (>30%)
        missing_cols = [col for col in X.columns if X[col].isna().mean() > 0.3]
        print(f"移除 {len(missing_cols)} 个高缺失特征: {missing_cols[:5]}...")
        
        
        #  批量处理相关性（避免一次性计算全矩阵）
        print("分批计算特征相关性...")
        corr_groups = {}
        
        # 按特征组分批处理
        for group in ['D', 'E', 'I', 'M', 'P', 'S', 'V', 'other']:
            group_cols = []
            if group == 'other':
                # 非标准组特征
                all_cols = set(X.columns)
                for g in ['D', 'E', 'I', 'M', 'P', 'S', 'V']:
                    pattern = re.compile(f'^{g}[0-9]+$', re.IGNORECASE)
                    group_cols = [c for c in X.columns if pattern.match(str(c))]
                    all_cols -= set(group_cols)
                group_cols = list(all_cols)
            else:
                pattern = re.compile(f'^{group}[0-9]+$', re.IGNORECASE)
                group_cols = [c for c in X.columns if pattern.match(str(c))]
            
            if not group_cols:
                continue
                
            print(f"  处理 {group} 组: {len(group_cols)} 个特征")
            
            # 仅计算组内相关性
            if len(group_cols) > 1:
                group_corr = X[group_cols].corr().abs()
                upper_tri = group_corr.where(np.triu(np.ones(group_corr.shape), k=1).astype(bool))
                high_corr_pairs = [(i,j) for i in range(len(group_cols)) for j in range(i+1, len(group_cols))
                                if upper_tri.iloc[i,j] > corr_threshold]
                
                to_drop = []
                for i,j in high_corr_pairs:
                    col_i = group_cols[i]
                    col_j = group_cols[j]
                    # 保留与目标相关性更高的特征
                    corr_i = abs(X[col_i].corr(y))
                    corr_j = abs(X[col_j].corr(y))
                    to_drop.append(col_j if corr_i > corr_j else col_i)
                
                corr_groups[group] = list(set(to_drop))
                print(f"    {group}组内移除 {len(corr_groups[group])} 个高度相关特征")
        
        # 合并需要移除的特征
        to_remove = constant_cols + missing_cols
        for group, cols in corr_groups.items():
            to_remove.extend(cols)
        to_remove = list(set(to_remove))
        
        X_filtered = X.drop(columns=to_remove)    
        return X_filtered
    
    def time_series_feature_importance(self, X, y, n_splits=5):
        """
        时序安全的特征重要性评估
        使用TimeSeriesSplit和LGBM计算稳健的特征重要性
        """
        X = X.copy()
        tscv = TimeSeriesSplit(n_splits=n_splits)
        
        # 初始化重要性容器
        feature_importances = pd.DataFrame(index=X.columns, columns=[f'fold_{i}' for i in range(n_splits)])
        fold_scores = []
        
        print("\n=== 时序安全特征重要性评估 ===")
        for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
            
            # 训练LGBM
            model = LGBMRegressor(
                n_estimators=200,
                learning_rate=0.05,
                num_leaves=10,
                random_state=42,
                
                min_data_in_leaf=5,          # 减少最小样本数，适合小数据集
                min_gain_to_split=0.0,       # 允许零增益分裂
                min_child_samples=1,         # 减少子节点最小样本              
                reg_alpha=0.1,               # L1正则化
                reg_lambda=0.1,              # L2正则化
                force_col_wise=True, 
                subsample=0.8,
                colsample_bytree=0.8,
                importance_type='split'  # 使用split而非gain，更稳定
            )
            
            # 修复: 使用回调函数替代early_stopping_rounds参数
            model.fit(
                X_train, y_train,
                eval_set=[(X_val, y_val)],
                eval_metric='rmse',  # 明确指定评估指标
                callbacks=[
                    lgb.early_stopping(stopping_rounds=20, verbose=False),
                    lgb.log_evaluation(period=0)  # 禁用日志输出
                ]
            )
            
            # 评估验证集性能
            val_pred = model.predict(X_val)
            rmse = np.sqrt(mean_squared_error(y_val, val_pred))
            fold_scores.append(rmse)
            print(f"Fold {fold+1} RMSE: {rmse:.6f}")
            
            # 存储特征重要性
            feature_importances[f'fold_{fold}'] = model.feature_importances_
        
        # 计算平均重要性
        feature_importances['mean_importance'] = feature_importances.mean(axis=1)
        feature_importances['std_importance'] = feature_importances.std(axis=1)
        
        # 归一化重要性
        feature_importances['normalized_importance'] = (
            feature_importances['mean_importance'] / 
            feature_importances['mean_importance'].sum()
        )
        
        # 排序
        feature_importances = feature_importances.sort_values('mean_importance', ascending=False)
        
        # 计算重要性稳定性 (标准差/均值)
        feature_importances['stability'] = (
            feature_importances['std_importance'] / 
            (feature_importances['mean_importance'] + 1e-8)
        )
        
        print(f"平均验证RMSE: {np.mean(fold_scores):.6f} ± {np.std(fold_scores):.6f}")
        self.feature_metadata['feature_importances'] = feature_importances
        
        return feature_importances
    
    def select_features_by_importance(self, feature_importances, max_features=100):
        #基于重要性选择特征，考虑稳定性和绝对重要性
        # 选择标准:
        # 1. 平均重要性 > 0
        # 2. 稳定性 < 1.0 (标准差不大于均值)
        # 3. 重要性排名在前max_features
        
         # 1. 宽松初步筛选 - 仅移除明显无效特征
        # (平均重要性 > 0 且 有基本稳定性)
        filtered_features = feature_importances[
            (feature_importances['mean_importance'] > 0) & 
            (feature_importances['stability'] < 3.0)  # 适度放宽稳定性阈值
        ]
        
        # 2. 按重要性排序
        sorted_features = filtered_features.sort_values('mean_importance', ascending=False)
        
        # 3. 选择前max_features个特征
        if len(sorted_features) > max_features:
            selected = sorted_features.head(max_features)
        else:
            selected = sorted_features
            print(f"警告: 仅找到 {len(selected)} 个符合基本条件的特征，小于目标 {max_features} 个")
            
        return selected.index.tolist()
    
    def rfecv_selection(self, X, y, initial_features, n_splits=5, min_features_to_select=30):
        """
        递归特征消除 (RFE) with cross-validation
        仅使用初始特征子集，大幅提高效率
        """
        
        print("\n=== 递归特征消除 (RFECV) ===")
        
        # 仅使用初始特征
        X_subset = X[initial_features]
        
        # 初始化LGBM评估器
        estimator = LGBMRegressor(
            n_estimators=200,
            learning_rate=0.05,
            num_leaves=10,
            random_state=42,
            min_data_in_leaf=5,          # 减少最小样本数，适合小数据集
            min_gain_to_split=0.0,       # 允许零增益分裂
            min_child_samples=1,         # 减少子节点最小样本
            colsample_bytree=0.8,        # 随机特征子集
            subsample=0.8,               # 随机样本子集
            reg_alpha=0.1,               # L1正则化
            reg_lambda=0.1,              # L2正则化
            force_col_wise=True          # 消除测试开销
        )
        
        tscv = TimeSeriesSplit(n_splits=n_splits)
    
        # 获取CV split的索引
        cv_indices = list(tscv.split(X_subset))
        
        # 执行RFECV
        selector = RFECV(
            estimator=estimator,
            step=0.1,  # 每次移除10%的特征
            cv=cv_indices,  # 直接传入分割索引
            scoring='neg_root_mean_squared_error',
            min_features_to_select=min_features_to_select,
            n_jobs=1,
            verbose=0
        )
        
        selector = selector.fit(X_subset, y)
        
        # 获取选择的特征
        selected_mask = selector.support_
        selected_features = np.array(initial_features)[selected_mask].tolist()
        
        print(f"RFECV选择特征数量: {len(selected_features)}")
        print(f"最佳特征数量: {selector.n_features_}")
        print(f"Top 20 RFECV特征: {selected_features[:20]}")
        
        # 保存RFECV结果
        self.feature_metadata['rfecv'] = {
            'selector': selector,
            'ranking': selector.ranking_,
            'cv_results': selector.cv_results_
        }
        
        return selected_features, selector
    
    def business_logic_validation(self, X, y, candidate_features):
        """
        业务逻辑验证
        1. 检查特征与目标的单调关系 (Spearman相关)
        2. 验证特征在不同市场环境下的稳定性
        3. 保留有金融意义的特征
        """
        print("\n=== 业务逻辑验证 ===")
        
        # 1. Spearman相关性检验
        spearman_corrs = {}
        for col in candidate_features:
            corr = X[col].corr(y, method='spearman')
            spearman_corrs[col] = abs(corr)
        
        # 2. 市场环境分组 (假设V_mean表示波动率)
        market_env_score = {}
        if 'V_mean' in X.columns:
            high_vol = X['V_mean'] > X['V_mean'].quantile(0.7)
            low_vol = ~high_vol
            
            for col in candidate_features:
                # 计算特征在不同环境下的重要性
                high_imp = abs(X.loc[high_vol, col].corr(y.loc[high_vol]))
                low_imp = abs(X.loc[low_vol, col].corr(y.loc[low_vol]))
                stability = abs(high_imp - low_imp) / (max(high_imp, low_imp) + 1e-8)
                market_env_score[col] = 1 - stability  # 稳定性得分 (1=完全稳定)
        
        # 3. 金融意义评分
        financial_score = {}
        for col in candidate_features:
            score = 0.0
            
            # 基础评分: 基于特征名识别金融意义
            if any(x in col.lower() for x in ['mean', 'std', 'cv', 'range']):
                score += 0.2  # 组统计特征
            
            if any(x in col.lower() for x in ['interaction', 'ratio', 'strength']):
                score += 0.3  # 交互特征
            
            if any(x in col.lower() for x in ['roll', 'lag']):
                score += 0.1  # 时序特征
            
            # 组合评分
            spearman = spearman_corrs.get(col, 0)
            market_stability = market_env_score.get(col, 0.5)
            financial_score[col] = 0.6 * spearman + 0.4 * (market_stability + score)
        
        # 4. 综合评分排序
        combined_scores = pd.Series(financial_score).sort_values(ascending=False)
        
        # 5. 选择Top特征 (至少30个)
        n_select = max(self.min_features, min(self.max_features, len(candidate_features)))
        final_features = combined_scores.index[:n_select].tolist()
        
        print(f"业务逻辑验证后选择 {len(final_features)} 个特征")
        print("Top 10 业务验证特征:", final_features[:10])
        
        # 保存验证结果
        self.feature_metadata['business_validation'] = {
            'spearman_corrs': spearman_corrs,
            'market_env_score': market_env_score,
            'financial_score': financial_score,
            'combined_scores': combined_scores
        }
        
        return final_features
    
    def fit(self, X, y):
        """
        完整特征选择流程
        X: 特征DataFrame
        y: 目标Series
        """
        print("=== 开始金融时间序列特征选择 ===")
        print(f"初始特征数: {X.shape[1]}")
        
        # 1. 预处理
        X_processed = self.preprocess_features(X)
        
        # 2. 预过滤
        X_filtered = self.pre_filter_features(X_processed, y)
        
        # 3. 重要性评估
        feature_importances = self.time_series_feature_importance(X_filtered, y)
        
        # 4. 重要性筛选 (先选100个)
        initial_features = self.select_features_by_importance(
            feature_importances, 
            max_features=100
        )
        
        # 5. RFECV (将100个减少到30-50个)
        rfecv_features, rfecv_selector = self.rfecv_selection(
            X_filtered, y, 
            initial_features,
            min_features_to_select=self.min_features
        )
        
        # 6. 业务逻辑验证 (最终选择30-50个)
        final_features = self.business_logic_validation(
            X_filtered, y, 
            rfecv_features
        )
        
        # 保存最终特征
        self.selected_features = final_features
        self.feature_metadata['final_features'] = final_features
        
        print(f"\n=== 特征选择完成 ===")
        print(f"最终选择特征数: {len(final_features)}")
        print(f"特征列表: {final_features}")
        
        return self
    
    def transform(self, X):
        """转换新数据，只保留选定特征"""
        if self.selected_features is None:
            raise ValueError("必须先调用fit()方法")
        
        # 只选择已选定的特征
        available_features = [f for f in self.selected_features if f in X.columns]
        missing_features = [f for f in self.selected_features if f not in X.columns]
        
        if missing_features:
            print(f"警告: {len(missing_features)} 个选定特征在新数据中不存在: {missing_features[:5]}...")
        
        return X[available_features]
    
    def plot_feature_importance(self, top_n=20):
        """绘制特征重要性"""
        if 'feature_importances' not in self.feature_metadata:
            print("未找到特征重要性数据")
            return
        
        importances = self.feature_metadata['feature_importances']
        top_features = importances.head(top_n)
        
        plt.figure(figsize=(12, 8))
        plt.barh(range(len(top_features)), top_features['mean_importance'].values[::-1])
        plt.yticks(range(len(top_features)), top_features.index[::-1])
        plt.xlabel('平均重要性')
        plt.title(f'Top {top_n} 特征重要性')
        plt.tight_layout()
        plt.savefig('feature_importance.png')
        print("特征重要性图已保存: 'feature_importance.png'")
    '''
     def plot_rfecv_results(self):
        """绘制RFECV结果"""
        if 'rfecv' not in self.feature_metadata:
            print("未找到RFECV数据")
            return
        
        rfecv_data = self.feature_metadata['rfecv']
        plt.figure(figsize=(10, 6))
        plt.xlabel("特征数量")
        plt.ylabel("交叉验证得分")
        plt.plot(range(1, len(rfecv_data['cv_results']) + 1), rfecv_data['cv_results'])
        plt.axvline(x=rfecv_data['selector'].n_features_, color="blue", linestyle="--")
        plt.title(f"RFECV: 最佳特征数 = {rfecv_data['selector'].n_features_}")
        plt.tight_layout()
        plt.savefig('rfecv_results.png')
        print("RFECV结果图已保存: 'rfecv_results.png'")
    '''
   


# 使用示例
if __name__ == "__main__":
    # 1. 初始化特征选择器
    selector = FinancialFeatureSelector(
        target_col=TARGET_COL,
        max_features=50,
        min_features=30
    )
    
    # 2. 应用特征选择
    print("\n=== 应用特征选择到训练集 ===")
    selector.fit(x_train, y_train)
    
    # 3. 转换训练集和测试集
    print("\n=== 转换数据集 ===")
    x_train_selected = selector.transform(x_train)
    x_test_selected = selector.transform(x_test)
    
    print(f"训练集特征数: {x_train_selected.shape[1]}")
    print(f"测试集特征数: {x_test_selected.shape[1]}")
    
    # 4. 可视化结果
    selector.plot_feature_importance(top_n=20)
    #selector.plot_rfecv_results()
    
    # 5. 保存选定特征
    selected_features = selector.selected_features
    feature_importances = selector.feature_metadata['feature_importances']
    
    # 保存特征重要性
    importance_df = feature_importances[['mean_importance', 'std_importance', 'stability']].copy()
    importance_df['rank'] = range(1, len(importance_df) + 1)
    importance_df.to_csv('feature_importances.csv')
    print("特征重要性已保存至 'feature_importances.csv'")
    
    # 保存选定特征列表
    with open('selected_features.txt', 'w') as f:
        f.write("\n".join(selected_features))
    print("选定特征列表已保存至 'selected_features.txt'")
    
    # 6. 业务逻辑验证报告
    print("\n=== 业务逻辑验证报告 ===")
    business_data = selector.feature_metadata['business_validation']
    top_features = selected_features[:10]
    
    print("\nTop 10 特征业务验证:")
    for i, feature in enumerate(top_features, 1):
        spearman = business_data['spearman_corrs'].get(feature, 0)
        stability = business_data['market_env_score'].get(feature, 0.5)
        score = business_data['financial_score'].get(feature, 0)
        
        print(f"{i}. {feature}:")
        print(f"   Spearman相关性: {spearman:.4f}")
        print(f"   市场环境稳定性: {stability:.4f}")
        print(f"   金融意义评分: {score:.4f}")
    
    # 7. 特征组分布分析
    print("\n=== 特征组分布分析 ===")
    group_distribution = {}
    for feature in selected_features:
        # 识别特征组 (基于命名约定)
        if feature.startswith('D_') or any(f'D{str(i)}' in feature for i in range(1, 10)):
            group = 'D'
        elif feature.startswith('E_') or any(f'E{str(i)}' in feature for i in range(1, 21)):
            group = 'E'
        elif feature.startswith('I_') or any(f'I{str(i)}' in feature for i in range(1, 10)):
            group = 'I'
        elif feature.startswith('M_') or any(f'M{str(i)}' in feature for i in range(1, 19)):
            group = 'M'
        elif feature.startswith('P_') or any(f'P{str(i)}' in feature for i in range(1, 14)):
            group = 'P'
        elif feature.startswith('S_') or any(f'S{str(i)}' in feature for i in range(1, 13)):
            group = 'S'
        elif feature.startswith('V_') or any(f'V{str(i)}' in feature for i in range(1, 14)):
            group = 'V'
        else:
            group = 'Other'
        
        group_distribution[group] = group_distribution.get(group, 0) + 1
    
    print("各组特征分布:")
    for group, count in group_distribution.items():
        print(f"  {group}组: {count}个 ({count/len(selected_features)*100:.1f}%)")



In [ ]:
#最新的详细特征工程选择代码
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import RFECV, SelectFromModel
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

class FinancialFeatureSelector:
    """
    金融时间序列安全的特征选择框架
    特点:
    1. 严格时间序列安全 (在TimeSeriesSplit内部执行所有特征选择)
    2. 分阶段筛选 (粗过滤->重要性排序->RFE->业务验证)
    3. 业务逻辑验证 (保留金融意义特征)
    4. 过拟合防护 (限制特征/样本比例)
    """
    
    def __init__(self, target_col='target', max_features=50, min_features=30):
        self.target_col = target_col
        self.max_features = max_features
        self.min_features = min_features
        self.selected_features = None
        self.feature_metadata = {}
    
    def preprocess_features(self, X):
        """处理特征，替换无穷大和填充缺失值"""
        X = X.copy()
        
        # 替换无穷大
        for col in X.select_dtypes(include=[np.number]).columns:
            X[col] = X[col].replace([np.inf, -np.inf], np.nan)
        
        # 用训练集统计数据填充缺失值
        for col in X.columns:
            if X[col].isna().any():
                if X[col].dtype in ['float32', 'float64']:
                    fill_value = X[col].median()
                else:
                    fill_value = X[col].mode()[0] if len(X[col].mode()) > 0 else 0
                X[col].fillna(fill_value, inplace=True)
        
        return X


    def pre_filter_features(self, X, y, corr_threshold=0.9):
        """
        预过滤: 移除明显无效特征
        1. 常量特征
        2. 高缺失率特征
        3. 高度相关特征
        """
        X = X.copy()
        
        # 1. 移除常量特征
        constant_cols = [col for col in X.columns if X[col].nunique() <= 1]
        print(f"移除 {len(constant_cols)} 个常量特征: {constant_cols[:5]}...")
        
        # 2. 移除高缺失率特征 (>30%)
        missing_cols = [col for col in X.columns if X[col].isna().mean() > 0.3]
        print(f"移除 {len(missing_cols)} 个高缺失特征: {missing_cols[:5]}...")
        
        
        #  批量处理相关性（避免一次性计算全矩阵）
        print("分批计算特征相关性...")
        corr_groups = {}
        
        # 按特征组分批处理
        for group in ['D', 'E', 'I', 'M', 'P', 'S', 'V', 'other']:
            group_cols = []
            if group == 'other':
                # 非标准组特征
                all_cols = set(X.columns)
                for g in ['D', 'E', 'I', 'M', 'P', 'S', 'V']:
                    pattern = re.compile(f'^{g}[0-9]+$', re.IGNORECASE)
                    group_cols = [c for c in X.columns if pattern.match(str(c))]
                    all_cols -= set(group_cols)
                group_cols = list(all_cols)
            else:
                pattern = re.compile(f'^{group}[0-9]+$', re.IGNORECASE)
                group_cols = [c for c in X.columns if pattern.match(str(c))]
            
            if not group_cols:
                continue
                
            print(f"  处理 {group} 组: {len(group_cols)} 个特征")
            
            # 仅计算组内相关性
            if len(group_cols) > 1:
                group_corr = X[group_cols].corr().abs()
                upper_tri = group_corr.where(np.triu(np.ones(group_corr.shape), k=1).astype(bool))
                high_corr_pairs = [(i,j) for i in range(len(group_cols)) for j in range(i+1, len(group_cols))
                                if upper_tri.iloc[i,j] > corr_threshold]
                
                to_drop = []
                for i,j in high_corr_pairs:
                    col_i = group_cols[i]
                    col_j = group_cols[j]
                    # 保留与目标相关性更高的特征
                    corr_i = abs(X[col_i].corr(y))
                    corr_j = abs(X[col_j].corr(y))
                    to_drop.append(col_j if corr_i > corr_j else col_i)
                
                corr_groups[group] = list(set(to_drop))
                print(f"    {group}组内移除 {len(corr_groups[group])} 个高度相关特征")
        
        # 合并需要移除的特征
        to_remove = constant_cols + missing_cols
        for group, cols in corr_groups.items():
            to_remove.extend(cols)
        to_remove = list(set(to_remove))

        X_filtered = X.drop(columns=to_remove)    
        return X_filtered
    
    def time_series_feature_importance(self, X, y, n_splits=5):
        """
        时序安全的特征重要性评估
        使用TimeSeriesSplit和LGBM计算稳健的特征重要性
        """
        X = X.copy()
        tscv = TimeSeriesSplit(n_splits=n_splits)
        
        # 初始化重要性容器
        feature_importances = pd.DataFrame(index=X.columns, columns=[f'fold_{i}' for i in range(n_splits)])
        fold_scores = []
        
        print("\n=== 时序安全特征重要性评估 ===")
        for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
            
            # 训练LGBM
            model = LGBMRegressor(
                n_estimators=100,
                learning_rate=0.1,
                num_leaves=31,
                min_child_samples=20,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                importance_type='split'  # 使用split而非gain，更稳定
            )
            
            # 修复: 使用回调函数替代early_stopping_rounds参数
            model.fit(
                X_train, y_train,
                eval_set=[(X_val, y_val)],
                eval_metric='rmse',  # 明确指定评估指标
                callbacks=[
                    lgb.early_stopping(stopping_rounds=20, verbose=False),
                    lgb.log_evaluation(period=0)  # 禁用日志输出
                ]
            )
            
            # 评估验证集性能
            val_pred = model.predict(X_val)
            rmse = np.sqrt(mean_squared_error(y_val, val_pred))
            fold_scores.append(rmse)
            print(f"Fold {fold+1} RMSE: {rmse:.6f}")
            
            # 存储特征重要性
            feature_importances[f'fold_{fold}'] = model.feature_importances_
        
        # 计算平均重要性
        feature_importances['mean_importance'] = feature_importances.mean(axis=1)
        feature_importances['std_importance'] = feature_importances.std(axis=1)
        
        # 归一化重要性
        feature_importances['normalized_importance'] = (
            feature_importances['mean_importance'] / 
            feature_importances['mean_importance'].sum()
        )
        
        # 排序
        feature_importances = feature_importances.sort_values('mean_importance', ascending=False)
        
        # 计算重要性稳定性 (标准差/均值)
        feature_importances['stability'] = (
            feature_importances['std_importance'] / 
            (feature_importances['mean_importance'] + 1e-8)
        )
        
        print(f"平均验证RMSE: {np.mean(fold_scores):.6f} ± {np.std(fold_scores):.6f}")
        self.feature_metadata['feature_importances'] = feature_importances
        
        return feature_importances
    
    def select_features_by_importance(self, feature_importances, max_features=100):
        #基于重要性选择特征，考虑稳定性和绝对重要性
        # 选择标准:
        # 1. 平均重要性 > 0
        # 2. 稳定性 < 1.0 (标准差不大于均值)
        # 3. 重要性排名在前max_features
        
         # 1. 宽松初步筛选 - 仅移除明显无效特征
        # (平均重要性 > 0 且 有基本稳定性)
        filtered_features = feature_importances[
            (feature_importances['mean_importance'] > 0) & 
            (feature_importances['stability'] < 3.0)  # 适度放宽稳定性阈值
        ]
        
        # 2. 按重要性排序
        sorted_features = filtered_features.sort_values('mean_importance', ascending=False)
        
        # 3. 选择前max_features个特征
        if len(sorted_features) > max_features:
            selected = sorted_features.head(max_features)
        else:
            selected = sorted_features
            print(f"警告: 仅找到 {len(selected)} 个符合基本条件的特征，小于目标 {max_features} 个")
            
        return selected.index.tolist()
    
    def rfecv_selection(self, X, y, initial_features, n_splits=5, min_features_to_select=30):
        """
        递归特征消除 (RFE) with cross-validation
        仅使用初始特征子集，大幅提高效率
        """
        
        print("\n=== 递归特征消除 (RFECV) ===")
        
        # 仅使用初始特征
        X_subset = X[initial_features]
        
        # 初始化LGBM评估器
        estimator = LGBMRegressor(
            n_estimators=100,
            learning_rate=0.1,
            num_leaves=31,
            random_state=42
        )
        
        tscv = TimeSeriesSplit(n_splits=n_splits)
    
        # 获取CV split的索引
        cv_indices = list(tscv.split(X_subset))
        
        # 执行RFECV
        selector = RFECV(
            estimator=estimator,
            step=0.1,  # 每次移除10%的特征
            cv=cv_indices,  # 直接传入分割索引
            scoring='neg_root_mean_squared_error',
            min_features_to_select=min_features_to_select,
            n_jobs=1,
            verbose=0
        )
        selector = selector.fit(X_subset, y)
        
        # 获取选择的特征
        selected_mask = selector.support_
        selected_features = np.array(initial_features)[selected_mask].tolist()
        
        print(f"RFECV选择特征数量: {len(selected_features)}")
        print(f"最佳特征数量: {selector.n_features_}")
        print(f"Top 20 RFECV特征: {selected_features[:20]}")
        
        # 保存RFECV结果
        self.feature_metadata['rfecv'] = {
            'selector': selector,
            'ranking': selector.ranking_,
            'cv_results': selector.cv_results_
        }
        
        return selected_features, selector
    
    def business_logic_validation(self, X, y, candidate_features):
        """
        业务逻辑验证
        1. 检查特征与目标的单调关系 (Spearman相关)
        2. 验证特征在不同市场环境下的稳定性
        3. 保留有金融意义的特征
        """
        print("\n=== 业务逻辑验证 ===")
        
        # 1. Spearman相关性检验
        spearman_corrs = {}
        for col in candidate_features:
            corr = X[col].corr(y, method='spearman')
            spearman_corrs[col] = abs(corr)
        
        # 2. 市场环境分组 (假设V_mean表示波动率)
        market_env_score = {}
        if 'V_mean' in X.columns:
            high_vol = X['V_mean'] > X['V_mean'].quantile(0.7)
            low_vol = ~high_vol
            
            for col in candidate_features:
                # 计算特征在不同环境下的重要性
                high_imp = abs(X.loc[high_vol, col].corr(y.loc[high_vol]))
                low_imp = abs(X.loc[low_vol, col].corr(y.loc[low_vol]))
                stability = abs(high_imp - low_imp) / (max(high_imp, low_imp) + 1e-8)
                market_env_score[col] = 1 - stability  # 稳定性得分 (1=完全稳定)
        
        # 3. 金融意义评分
        financial_score = {}
        for col in candidate_features:
            score = 0.0
            
            # 基础评分: 基于特征名识别金融意义
            if any(x in col.lower() for x in ['mean', 'std', 'cv', 'range']):
                score += 0.2  # 组统计特征
            
            if any(x in col.lower() for x in ['interaction', 'ratio', 'strength']):
                score += 0.3  # 交互特征
            
            if any(x in col.lower() for x in ['roll', 'lag']):
                score += 0.1  # 时序特征
            
            # 组合评分
            spearman = spearman_corrs.get(col, 0)
            market_stability = market_env_score.get(col, 0.5)
            financial_score[col] = 0.6 * spearman + 0.4 * (market_stability + score)
        
        # 4. 综合评分排序
        combined_scores = pd.Series(financial_score).sort_values(ascending=False)
        
        # 5. 选择Top特征 (至少30个)
        n_select = max(self.min_features, min(self.max_features, len(candidate_features)))
        final_features = combined_scores.index[:n_select].tolist()
        
        print(f"业务逻辑验证后选择 {len(final_features)} 个特征")
        print("Top 10 业务验证特征:", final_features[:10])
        
        # 保存验证结果
        self.feature_metadata['business_validation'] = {
            'spearman_corrs': spearman_corrs,
            'market_env_score': market_env_score,
            'financial_score': financial_score,
            'combined_scores': combined_scores
        }
        
        return final_features
    
    def fit(self, X, y):
        """
        完整特征选择流程
        X: 特征DataFrame
        y: 目标Series
        """
        print("=== 开始金融时间序列特征选择 ===")
        print(f"初始特征数: {X.shape[1]}")
        
        # 1. 预处理
        X_processed = self.preprocess_features(X)
        
        # 2. 预过滤
        X_filtered = self.pre_filter_features(X_processed, y)
        
        # 3. 重要性评估
        feature_importances = self.time_series_feature_importance(X_filtered, y)
        
        # 4. 重要性筛选 (先选100个)
        initial_features = self.select_features_by_importance(
            feature_importances, 
            max_features=100
        )
        
        # 5. RFECV (将100个减少到30-50个)
        rfecv_features, rfecv_selector = self.rfecv_selection(
            X_filtered, y, 
            initial_features,
            min_features_to_select=self.min_features
        )
        
        # 6. 业务逻辑验证 (最终选择30-50个)
        final_features = self.business_logic_validation(
            X_filtered, y, 
            rfecv_features
        )
        
        # 保存最终特征
        self.selected_features = final_features
        self.feature_metadata['final_features'] = final_features
        
        print(f"\n=== 特征选择完成 ===")
        print(f"最终选择特征数: {len(final_features)}")
        print(f"特征列表: {final_features}")
        
        return self
    
    def transform(self, X):
        """转换新数据，只保留选定特征"""
        if self.selected_features is None:
            raise ValueError("必须先调用fit()方法")
        
        # 只选择已选定的特征
        available_features = [f for f in self.selected_features if f in X.columns]
        missing_features = [f for f in self.selected_features if f not in X.columns]
        
        if missing_features:
            print(f"警告: {len(missing_features)} 个选定特征在新数据中不存在: {missing_features[:5]}...")
        
        return X[available_features]
    
    def plot_feature_importance(self, top_n=20):
        """绘制特征重要性"""
        if 'feature_importances' not in self.feature_metadata:
            print("未找到特征重要性数据")
            return
        
        importances = self.feature_metadata['feature_importances']
        top_features = importances.head(top_n)
        
        plt.figure(figsize=(12, 8))
        plt.barh(range(len(top_features)), top_features['mean_importance'].values[::-1])
        plt.yticks(range(len(top_features)), top_features.index[::-1])
        plt.xlabel('平均重要性')
        plt.title(f'Top {top_n} 特征重要性')
        plt.tight_layout()
        plt.savefig('feature_importance.png')
        print("特征重要性图已保存: 'feature_importance.png'")
    '''
    def plot_rfecv_results(self):
        """绘制RFECV结果"""
        if 'rfecv' not in self.feature_metadata:
            print("未找到RFECV数据")
            return
        
        rfecv_data = self.feature_metadata['rfecv']
        plt.figure(figsize=(10, 6))
        plt.xlabel("特征数量")
        plt.ylabel("交叉验证得分")
        plt.plot(range(1, len(rfecv_data['cv_results']) + 1), rfecv_data['cv_results'])
        plt.axvline(x=rfecv_data['selector'].n_features_, color="blue", linestyle="--")
        plt.title(f"RFECV: 最佳特征数 = {rfecv_data['selector'].n_features_}")
        plt.tight_layout()
        plt.savefig('rfecv_results.png')
        print("RFECV结果图已保存: 'rfecv_results.png'")
'''

# 使用示例
if __name__ == "__main__":
    # 1. 初始化特征选择器
    selector = FinancialFeatureSelector(
        target_col=TARGET_COL,
        max_features=50,
        min_features=30
    )
    
    # 2. 应用特征选择
    print("\n=== 应用特征选择到训练集 ===")
    selector.fit(x_train, y_train)
    
    # 3. 转换训练集和测试集
    print("\n=== 转换数据集 ===")
    x_train_selected = selector.transform(x_train)
    x_test_selected = selector.transform(x_test)
    
    print(f"训练集特征数: {x_train_selected.shape[1]}")
    print(f"测试集特征数: {x_test_selected.shape[1]}")
    
    # 4. 可视化结果
    selector.plot_feature_importance(top_n=20)
    selector.plot_rfecv_results()
    
    # 5. 保存选定特征
    selected_features = selector.selected_features
    feature_importances = selector.feature_metadata['feature_importances']
    
    # 保存特征重要性
    importance_df = feature_importances[['mean_importance', 'std_importance', 'stability']].copy()
    importance_df['rank'] = range(1, len(importance_df) + 1)
    importance_df.to_csv('feature_importances.csv')
    print("特征重要性已保存至 'feature_importances.csv'")
    
    # 保存选定特征列表
    with open('selected_features.txt', 'w') as f:
        f.write("\n".join(selected_features))
    print("选定特征列表已保存至 'selected_features.txt'")
    
    # 6. 业务逻辑验证报告
    print("\n=== 业务逻辑验证报告 ===")
    business_data = selector.feature_metadata['business_validation']
    top_features = selected_features[:10]
    
    print("\nTop 10 特征业务验证:")
    for i, feature in enumerate(top_features, 1):
        spearman = business_data['spearman_corrs'].get(feature, 0)
        stability = business_data['market_env_score'].get(feature, 0.5)
        score = business_data['financial_score'].get(feature, 0)
        
        print(f"{i}. {feature}:")
        print(f"   Spearman相关性: {spearman:.4f}")
        print(f"   市场环境稳定性: {stability:.4f}")
        print(f"   金融意义评分: {score:.4f}")
    
    # 7. 特征组分布分析
    print("\n=== 特征组分布分析 ===")
    group_distribution = {}
    for feature in selected_features:
        # 识别特征组 (基于命名约定)
        if feature.startswith('D_') or any(f'D{str(i)}' in feature for i in range(1, 10)):
            group = 'D'
        elif feature.startswith('E_') or any(f'E{str(i)}' in feature for i in range(1, 21)):
            group = 'E'
        elif feature.startswith('I_') or any(f'I{str(i)}' in feature for i in range(1, 10)):
            group = 'I'
        elif feature.startswith('M_') or any(f'M{str(i)}' in feature for i in range(1, 19)):
            group = 'M'
        elif feature.startswith('P_') or any(f'P{str(i)}' in feature for i in range(1, 14)):
            group = 'P'
        elif feature.startswith('S_') or any(f'S{str(i)}' in feature for i in range(1, 13)):
            group = 'S'
        elif feature.startswith('V_') or any(f'V{str(i)}' in feature for i in range(1, 14)):
            group = 'V'
        else:
            group = 'Other'
        
        group_distribution[group] = group_distribution.get(group, 0) + 1
    
    print("各组特征分布:")
    for group, count in group_distribution.items():
        print(f"  {group}组: {count}个 ({count/len(selected_features)*100:.1f}%)")

# 8. 后续使用
print("\n=== 后续使用建议 ===")
print("1. 用选定特征训练最终模型:")
print("   model = LGBMRegressor(n_estimators=500, learning_rate=0.01)")
print("   model.fit(x_train_selected, y_train)")
print("")
print("2. 特征重要性监控:")
print("   - 定期检查特征在新数据上的表现")
print("   - 当模型性能下降时，重新运行特征选择")
print("")
print("3. 部署注意事项:")
print("   - 确保生产环境特征计算与训练环境完全一致")
print("   - 实现特征失效的降级策略 (如用历史均值替代)")
print("")
print("4. 特征扩展建议:")
print("   - 优先探索选定特征的非线性变换 (如平方、对数)")
print("   - 考虑选定特征的交互项，而非添加新原始特征")


In [ ]:
from sklearn.feature_selection import VarianceThreshold, mutual_info_regression
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNetCV
import xgboost as xgb

# ===== 1. 过滤式初筛 =====
var_selector = VarianceThreshold(threshold=0.01).fit(x_train)
x_train_var = pd.DataFrame(var_selector.transform(x_train), columns=x_train.columns[var_selector.get_support()])
x_test_var = pd.DataFrame(var_selector.transform(x_test), columns=x_test.columns[var_selector.get_support()])

# 互信息选择 (在原始尺度上计算!)
mi_scores = mutual_info_regression(x_train_var, y_train, random_state=42)
top_features = pd.Series(mi_scores, index=x_train_var.columns).sort_values(ascending=False).index[:50].tolist()

# ===== 2. 标准化选定特征 =====
scaler = StandardScaler().fit(x_train_var[top_features])  # 仅拟合选定的50个特征
x_train_scaled = pd.DataFrame(scaler.transform(x_train_var[top_features]), columns=top_features)
x_test_scaled = pd.DataFrame(scaler.transform(x_test_var[top_features]), columns=top_features)
y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).flatten()

# ===== 3. ElasticNet + XGBoost 融合 =====
# ElasticNet (抗共线性)
elastic = ElasticNetCV(l1_ratio=[0.1,0.3, 0.5,0.7, 0.9], cv=3, max_iter=10000,random_state=42).fit(x_train_scaled, y_train)
elastic_features = x_train_scaled.columns[np.abs(elastic.coef_) > 1e-5].tolist()

# XGBoost (非线性)
xgb_model = xgb.XGBRegressor(n_estimators=200, learning_rate=0.05, random_state=42).fit(x_train_scaled, y_train)
xgb_features = pd.Series(xgb_model.feature_importances_, index=top_features).sort_values(ascending=False).index[:40].tolist()

# 融合选择Top30
candidate_features = list(set(elastic_features + xgb_features))
rank_df = pd.DataFrame(index=candidate_features)
rank_df['elastic_rank'] = pd.Series(elastic.coef_, index=top_features).reindex(candidate_features).abs().rank(ascending=False)
rank_df['xgb_rank'] = pd.Series(xgb_model.feature_importances_, index=top_features).reindex(candidate_features).rank(ascending=False)
rank_df['final_rank'] = 0.6 * rank_df['elastic_rank'] + 0.4 * rank_df['xgb_rank']
top30_features = rank_df.sort_values('final_rank').index[:30].tolist()

# ===== 4. 应用最终特征 =====
x_train_final = x_train_scaled[top30_features]
x_test_final = x_test_scaled[top30_features]

print(f"✅ 特征选择完成 | 最终特征: {len(top30_features)}")
print(f"Top5: {top30_features[:5]}")

# 保存筛选后数据集 (保持时间顺序!)
filtered_train = pd.concat([x_train_final, y_train], axis=1)
filtered_test = pd.concat([x_test_final, y_test], axis=1)
filtered_all = pd.concat([filtered_train, filtered_test])

模型拟合

In [ ]:
n= len(x_train)
split_idx = int(0.85 * n)  # 85% 训练，15% 验证（用于早停）

X_tr = x_train.iloc[:split_idx]
X_val = x_train.iloc[split_idx:]

y_tr = y_train.iloc[:split_idx]
y_val = y_train.iloc[split_idx:]
val_forward_returns=train_score_forward_returns[split_idx:]
val_risk_free_rate=train_score_risk_free_rate[split_idx:]
# ----------------------------
# 2. 定义 LightGBM 模型
# ----------------------------
params = {
    'objective': 'regression',
    'metric': 'mse',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'verbose': -1,
    'random_state': 42,
    'n_estimators': 2000  # 将被早停截断
}

train_data = lgb.Dataset(X_tr, label=y_tr)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

# ----------------------------
# 3. 训练模型（带早停）
# ----------------------------
model = lgb.train(
    params,
    train_data,
    valid_sets=[val_data],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=True),
        lgb.log_evaluation(100)
    ]
)

# ----------------------------
# 4. 预测（raw output）
# ----------------------------
# 对训练集 & 测试集都预测（方便后续分析）
pred_train_raw = model.predict(x_train)   # 用于诊断
pred_test_raw = model.predict(x_test)     # 最终输出（待后处理）

# 转为 Series（保留索引）
pred_test_raw = pd.Series(pred_test_raw, index=x_test.index, name='pred_excess')

预测值映射成仓位以及预测函数score的变种夏普比率

In [ ]:
# ----------------------------
# Step 0: 内联你提供的 score 函数（避免依赖）
# ----------------------------
MIN_INVESTMENT = 0.00
MAX_INVESTMENT = 2.00

class ParticipantVisibleError(Exception):
    pass

def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str = None) -> float:
    if not pd.api.types.is_numeric_dtype(submission['prediction']):
        raise ParticipantVisibleError('Predictions must be numeric')
    
    # 使用 copy 避免修改原始数据
    sol = solution.copy()
    sol['position'] = submission['prediction']
    
    if sol['position'].max() > MAX_INVESTMENT:
        raise ParticipantVisibleError(f'Position of {sol["position"].max()} exceeds maximum of {MAX_INVESTMENT}')
    if sol['position'].min() < MIN_INVESTMENT:
        raise ParticipantVisibleError(f'Position of {sol["position"].min()} below minimum of {MIN_INVESTMENT}')
    
    sol['strategy_returns'] = sol['risk_free_rate'] * (1 - sol['position']) + sol['position'] * sol['forward_returns']
    
    # Strategy Sharpe
    strategy_excess_returns = sol['strategy_returns'] - sol['risk_free_rate']
    n = len(sol)
    strategy_excess_cumulative = (1 + strategy_excess_returns).prod()
    strategy_mean_excess_return = strategy_excess_cumulative ** (1 / n) - 1
    strategy_std = sol['strategy_returns'].std()
    
    trading_days_per_yr = 252
    if strategy_std == 0:
        raise ParticipantVisibleError('Division by zero, strategy std is zero')
    sharpe = strategy_mean_excess_return / strategy_std * np.sqrt(trading_days_per_yr)
    
    # Market stats
    market_excess_returns = sol['forward_returns'] - sol['risk_free_rate']
    market_excess_cumulative = (1 + market_excess_returns).prod()
    market_mean_excess_return = market_excess_cumulative ** (1 / n) - 1
    market_std = sol['forward_returns'].std()
    market_volatility = float(market_std * np.sqrt(trading_days_per_yr) * 100)
    
    if market_volatility == 0:
        raise ParticipantVisibleError('Division by zero, market std is zero')
    
    # Volatility penalty
    strategy_volatility = float(strategy_std * np.sqrt(trading_days_per_yr) * 100)
    excess_vol = max(0, strategy_volatility / market_volatility - 1.2)
    vol_penalty = 1 + excess_vol
    
    # Return penalty
    return_gap = max(0, (market_mean_excess_return - strategy_mean_excess_return) * 100 * trading_days_per_yr)
    return_penalty = 1 + (return_gap ** 2) / 100
    
    adjusted_sharpe = sharpe / (vol_penalty * return_penalty)
    return min(float(adjusted_sharpe), 1_000_000)

# ----------------------------
# Step 1: 仓位映射函数（可调）
# ----------------------------
def map_to_position(pred_excess: np.ndarray, method='sigmoid', k=10.0):
    """
    将预测的 excess return 映射到 [0, 2] 仓位。
    
    Parameters:
    - pred_excess: array of raw model predictions (y_hat)
    - method: 'sigmoid', 'clip_scale', or 'binary'
    - k: sigmoid 陡峭度（越大越接近阶跃）
    """
    if method == 'sigmoid':
        # Sigmoid 映射到 (0, 2)
        position = 2 / (1 + np.exp(-k * pred_excess))
    elif method == 'clip_scale':
        # 截断后线性缩放到 [0, 2]
        lower = np.percentile(pred_excess, 5)
        upper = np.percentile(pred_excess, 95)
        clipped = np.clip(pred_excess, lower, upper)
        position = 2 * (clipped - clipped.min()) / (clipped.max() - clipped.min() + 1e-8)
    elif method == 'binary':
        # 简单二值：正收益满仓，否则空仓
        position = np.where(pred_excess > 0, 2.0, 0.0)
    else:
        raise ValueError("method must be 'sigmoid', 'clip_scale', or 'binary'")
    
    # 确保严格在 [0, 2]
    position = np.clip(position, 0, 2)
    return position

# ----------------------------
# Step 2: 完整评估流程
# ----------------------------
def evaluate_model(
    pred_test_raw: pd.Series,
    forward_returns: pd.Series,
    risk_free_rate: pd.Series,
    mapping_method='sigmoid',
    k=10.0
) -> tuple[float, pd.Series]:
    """
    输入：
        pred_test_raw: 模型对 x_test 的原始预测 (y_hat)
        forward_returns: 对应的真实 forward_returns
        risk_free_rate: 对应的 risk_free_rate
    
    输出：
        adjusted_sharpe: 改进夏普比率
        position: 映射后的仓位（可用于分析）
    """
    # 确保索引对齐
    assert pred_test_raw.index.equals(forward_returns.index), "Index mismatch"
    assert pred_test_raw.index.equals(risk_free_rate.index), "Index mismatch"
    
    # Step A: 映射到仓位
    position = map_to_position(
        pred_test_raw.values,
        method=mapping_method,
        k=k
    )
    position = pd.Series(position, index=pred_test_raw.index, name='prediction')
    
    # Step B: 构造 solution 和 submission
    solution_df = pd.DataFrame({
        'forward_returns': forward_returns,
        'risk_free_rate': risk_free_rate
    })
    submission_df = pd.DataFrame({'prediction': position})
    
    # Step C: 调用 score
    try:
        adj_sharpe = score(solution_df, submission_df, row_id_column_name=None)
    except ParticipantVisibleError as e:
        print(f"Score error: {e}")
        adj_sharpe = -np.inf
    
    return adj_sharpe, position

# ----------------------------
# 示例用法（假设你已有以下变量）：
# - pred_test_raw: pd.Series
# - forward_returns_test: pd.Series (对应 x_test)
# - risk_free_rate_test: pd.Series (对应 x_test)
# ----------------------------
# adjusted_sharpe, position = evaluate_model(
#     pred_test_raw,
#     forward_returns_test,
#     risk_free_rate_test,
#     mapping_method='sigmoid',
#     k=15.0
# )
# print(f"Adjusted Sharpe: {adjusted_sharpe:.4f}")

进行各个模型的验证和调参，确定最优模型与最优参数

In [ ]:
# ------------------------------------------------------------------
# 3. 消融实验配置
# ------------------------------------------------------------------
# 超参网格（可按需扩展）
lgb_param_grid = [
    {'learning_rate': 0.05, 'num_leaves': 31, 'feature_fraction': 0.8},
    {'learning_rate': 0.1,  'num_leaves': 31, 'feature_fraction': 0.8},
    {'learning_rate': 0.05, 'num_leaves': 63, 'feature_fraction': 0.7},
    {'learning_rate': 0.01, 'num_leaves': 15, 'feature_fraction': 0.9},
]

mapping_methods = ['sigmoid', 'clip_scale', 'binary']
k_values = [5, 10, 15, 20]  # 仅 sigmoid 使用

# 存储结果
results = []
best_score = -np.inf
best_config = None

# ------------------------------------------------------------------
# 4. 实验循环
# ------------------------------------------------------------------
for i, params in enumerate(lgb_param_grid):
    print(f"\n[Experiment {i+1}/{len(lgb_param_grid)}] Training with params: {params}")
    
    # 构建 LightGBM Dataset
    train_data = lgb.Dataset(X_tr, label=y_tr)
    val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
    
    # 固定其他参数
    fixed_params = {
        'objective': 'regression',
        'metric': 'mse',
        'boosting_type': 'gbdt',
        'bagging_fraction': 0.8,
        'bagging_freq': 1,
        'verbose': -1,
        'random_state': 42,
        'n_estimators': 2000
    }
    full_params = {**fixed_params, **params}
    
    # 训练模型（带早停）
    model = lgb.train(
        full_params,
        train_data,
        valid_sets=[val_data],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(0)  # 静默
        ]
    )
    
    # 预测验证集 raw output
    pred_val_raw = model.predict(X_val)
    pred_val_raw = pd.Series(pred_val_raw, index=X_val.index)
    
    # 遍历映射策略
    for method in mapping_methods:
        if method == 'sigmoid':
            for k in k_values:
                position = map_to_position(pred_val_raw.values, method=method, k=k)
                submission_df = pd.DataFrame({'prediction': position}, index=X_val.index)
                solution_df = pd.DataFrame({
                    'forward_returns': val_forward_returns.values,
                    'risk_free_rate': val_risk_free_rate.values
                }, index=X_val.index)
                
                try:
                    val_score = score(solution_df, submission_df, row_id_column_name=None)
                except ParticipantVisibleError:
                    val_score = -np.inf
                
                # 记录
                config = {
                    'lgb_params': params,
                    'mapping_method': method,
                    'k': k,
                    'val_score': val_score
                }
                results.append(config)
                
                if val_score > best_score:
                    best_score = val_score
                    best_config = config.copy()
                    best_config['model'] = model  # 保留模型引用
                
                print(f"  → method={method}, k={k} → Adjusted Sharpe: {val_score:.4f}")
        else:
            position = map_to_position(pred_val_raw.values, method=method)
            submission_df = pd.DataFrame({'prediction': position}, index=X_val.index)
            solution_df = pd.DataFrame({
                'forward_returns': val_forward_returns.values,
                'risk_free_rate': val_risk_free_rate.values
            }, index=X_val.index)
            
            try:
                val_score = score(solution_df, submission_df, row_id_column_name=None)
            except ParticipantVisibleError:
                val_score = -np.inf
            
            config = {
                'lgb_params': params,
                'mapping_method': method,
                'k': None,
                'val_score': val_score
            }
            results.append(config)
            
            if val_score > best_score:
                best_score = val_score
                best_config = config.copy()
                best_config['model'] = model
            
            print(f"  → method={method} → Adjusted Sharpe: {val_score:.4f}")

# ------------------------------------------------------------------
# 5. 输出最佳结果
# ------------------------------------------------------------------
print("\n" + "="*60)
print("✅ Best Configuration on Validation Set:")
print(f"Adjusted Sharpe: {best_score:.4f}")
print(f"Model Params: {best_config['lgb_params']}")
print(f"Mapping Method: {best_config['mapping_method']}")
if best_config['mapping_method'] == 'sigmoid':
    print(f"Sigmoid k: {best_config['k']}")
print("="*60)

# 保存 results 到 DataFrame（可选）
results_df = pd.DataFrame(results)
results_df.to_csv('ablation_results.csv', index=False)
print("\nSaved all results to 'ablation_results.csv'")

验证情况与可视化

In [ ]:
from datetime import datetime
# ------------------------------------------------------------------
# 1. 确保 best_config 已从消融实验获取
# ------------------------------------------------------------------
# 假设 best_config 已存在，包含:
# - best_config['lgb_params']: 最佳超参
# - best_config['mapping_method']: 最佳映射方法
# - best_config['k']: 最佳k值（如果是sigmoid）
print(f"Best config from validation: {best_config}")

# ------------------------------------------------------------------
# 2. 用全训练集重新训练最终模型
# ------------------------------------------------------------------
print("\n🚀 Training final model on full training set...")

# 准备全训练集数据
train_data_full = lgb.Dataset(x_train, label=y_train)

# 合并固定参数 + 最佳参数
fixed_params = {
    'objective': 'regression',
    'metric': 'mse',
    'boosting_type': 'gbdt',
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'verbose': -1,
    'random_state': 42,
    'n_estimators': 2000
}
final_params = {**fixed_params, **best_config['lgb_params']}

# 为早停准备一个小的内部验证集（取训练集最后10%）
n = len(x_train)
val_size = int(0.1 * n)
X_internal_val = x_train.iloc[-val_size:]
y_internal_val = y_train.iloc[-val_size:]
val_data_internal = lgb.Dataset(X_internal_val, label=y_internal_val, reference=train_data_full)

# 训练最终模型（带早停）
final_model = lgb.train(
    final_params,
    train_data_full,
    valid_sets=[val_data_internal],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=True),
        lgb.log_evaluation(50)
    ]
)

# 保存模型（可选）
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_path = f'final_model_{timestamp}.pkl'
joblib.dump(final_model, model_path)
print(f"💾 Model saved to {model_path}")

# ------------------------------------------------------------------
# 3. 测试集预测
# ------------------------------------------------------------------
print("\n🔍 Predicting on test set...")
pred_test_raw = final_model.predict(x_test)
pred_test_raw = pd.Series(pred_test_raw, index=x_test.index, name='pred_excess')

# ------------------------------------------------------------------
# 4. 仓位映射（使用最佳配置）
# ------------------------------------------------------------------
print("\n🔄 Mapping predictions to positions...")
if best_config['mapping_method'] == 'sigmoid':
    position_test = map_to_position(
        pred_test_raw.values,
        method='sigmoid',
        k=best_config['k']
    )
else:
    position_test = map_to_position(
        pred_test_raw.values,
        method=best_config['mapping_method']
    )

position_test = pd.Series(position_test, index=x_test.index, name='position')
print(f"Position stats: min={position_test.min():.4f}, max={position_test.max():.4f}, mean={position_test.mean():.4f}")

# ------------------------------------------------------------------
# 5. 构造 solution 和 submission 用于 score
# ------------------------------------------------------------------
# 确保你有测试集对应的 forward_returns_test 和 risk_free_rate_test
# 假设它们已经存在且与 x_test 索引对齐

solution_test = pd.DataFrame({
    'forward_returns': score_forward_returns,
    'risk_free_rate': score_risk_free_rate
}, index=x_test.index)

submission_test = pd.DataFrame({
    'prediction': position_test
}, index=x_test.index)

# ------------------------------------------------------------------
# 6. 调用 score 函数计算最终分数
# ------------------------------------------------------------------
print("\n📊 Calculating final Adjusted Sharpe Ratio...")
try:
    final_score = score(solution_test, submission_test, row_id_column_name=None)
    print(f"\n🎉 FINAL TEST ADJUSTED SHARPE RATIO: {final_score:.4f}")
except ParticipantVisibleError as e:
    print(f"❌ Score calculation failed: {e}")
    final_score = -np.inf

# ------------------------------------------------------------------
# 7. 保存结果与分析
# ------------------------------------------------------------------
# 7.1 保存仓位
positions_df = pd.DataFrame({
    'date': x_test.index,  # 假设索引是日期
    'position': position_test,
    'pred_excess': pred_test_raw,
    'forward_returns': score_forward_returns,
    'risk_free_rate': score_risk_free_rate
})
positions_df.to_csv(f'test_positions_{timestamp}.csv', index=False)
print(f"💾 Positions saved to test_positions_{timestamp}.csv")

# 7.2 保存最终分数
with open(f'final_score_{timestamp}.txt', 'w') as f:
    f.write(f"Final Adjusted Sharpe Ratio: {final_score:.4f}\n")
    f.write(f"Model params: {final_params}\n")
    f.write(f"Mapping method: {best_config['mapping_method']}\n")
    if best_config['mapping_method'] == 'sigmoid':
        f.write(f"Sigmoid k: {best_config['k']}\n")
print(f"💾 Score report saved to final_score_{timestamp}.txt")


def plot_strategy_analysis(positions_df, final_score):
    """生成策略分析图表"""
    plt.figure(figsize=(15, 10))
    
    # 2. 策略收益 vs 市场收益
    positions_df['strategy_returns'] = (
        positions_df['risk_free_rate'] * (1 - positions_df['position']) + 
        positions_df['position'] * positions_df['forward_returns']
    )
    positions_df['market_returns'] = positions_df['forward_returns']
    
    # 计算累计收益
    positions_df['cum_strategy'] = (1 + positions_df['strategy_returns']).cumprod()
    positions_df['cum_market'] = (1 + positions_df['market_returns']).cumprod()
    
    plt.subplot(3, 1, 2)
    plt.plot(positions_df['date'], positions_df['cum_strategy'], label='Strategy', linewidth=2)
    plt.plot(positions_df['date'], positions_df['cum_market'], label='Market (S&P 500)', linestyle='--')
    plt.title('Cumulative Returns: Strategy vs Market', fontsize=14)
    plt.ylabel('Cumulative Return')
    plt.legend()
    plt.grid(alpha=0.3)
    
    # 3. 仓位 vs 预测信号
    plt.subplot(3, 1, 3)
    plt.scatter(positions_df['pred_excess'], positions_df['position'], 
                alpha=0.6, s=10, color='purple')
    plt.title('Position vs Predicted Excess Return', fontsize=14)
    plt.xlabel('Predicted Excess Return')
    plt.ylabel('Position')
    plt.grid(alpha=0.3)
    
    plt.tight_layout()
    plot_path = f'strategy_analysis_{timestamp}.png'
    plt.savefig(plot_path, dpi=120, bbox_inches='tight')
    print(f"📈 Strategy analysis plot saved to {plot_path}")
    plt.close()

# 生成分析图表
try:
    plot_strategy_analysis(positions_df, final_score)
except Exception as e:
    print(f"⚠️ Failed to generate plots: {e}")

# ------------------------------------------------------------------
# 9. 特征重要性分析（可选）
# ------------------------------------------------------------------
if hasattr(final_model, 'feature_importance'):
    # 获取特征重要性
    feature_importance = pd.DataFrame({
        'feature': x_train.columns,
        'importance': final_model.feature_importance(importance_type='gain')
    }).sort_values('importance', ascending=False)
    
    # 保存
    feature_importance.to_csv(f'feature_importance_{timestamp}.csv', index=False)
    print(f"📊 Feature importance saved to feature_importance_{timestamp}.csv")
    
    # 绘制Top 15
    plt.figure(figsize=(12, 8))
    sns.barplot(
        x='importance', 
        y='feature', 
        data=feature_importance.head(15),
        palette='viridis'
    )
    plt.title('Top 15 Features by Importance (Gain)', fontsize=14)
    plt.tight_layout()
    plt.savefig(f'feature_importance_plot_{timestamp}.png', dpi=120)
    plt.close()
    print(f"📈 Feature importance plot saved")

# ------------------------------------------------------------------
# 10. 完成提示
# ------------------------------------------------------------------
print("\n" + "="*60)
print("✅ FINAL EVALUATION COMPLETE!")
print(f"🎯 Final Test Adjusted Sharpe Ratio: {final_score:.4f}")
print(f"💾 All results saved with timestamp: {timestamp}")
print("="*60)

launch server

In [ ]:
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))